# CNN

We start by importing the libraries.

In [ ]:
# Import libraries
import numpy as np
import time
import matplotlib
import matplotlib.image as mpimg
import pandas as pd
from torch import optim
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
nltk.download('wordnet')
from sklearn.feature_extraction.text import TfidfTransformer
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('punkt')
from sklearn.feature_extraction.text import CountVectorizer
import re
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
import spacy
from sklearn.preprocessing import OneHotEncoder
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout, SpatialDropout1D, Conv1D, MaxPooling1D, GRU, BatchNormalization
from tensorflow.keras.layers import Input, Bidirectional, GlobalAveragePooling1D, concatenate, LeakyReLU, GlobalMaxPooling1D, Flatten
from tensorflow.keras import regularizers
from tensorflow.keras import backend as K
import torch
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

We also import the data.

In [ ]:
# Train data 
traindata_all = pd.read_csv('/content/drive/MyDrive/TFM/multimodal_train.tsv',sep='\t')
# Validation data 
validata_all = pd.read_csv('/content/drive/MyDrive/TFM/multimodal_validate.tsv',sep='\t')
# Test data 
testdata_all = pd.read_csv('/content/drive/MyDrive/TFM/multimodal_test_public.tsv',sep='\t')

We select a subset of the dataframe with no missing values in the 'clean_title' column.

In [ ]:
# Train data with no missing values
train_data = traindata_all[traindata_all['clean_title'].notnull().to_numpy()]
# Validation data with no missing values
valid_data = validata_all[validata_all['clean_title'].notnull().to_numpy()]
# Test data with no missing values
test_data = testdata_all[testdata_all['clean_title'].notnull().to_numpy()]

And we separate the datasets into the texts and the labels

In [ ]:
## Train data
train_news = list(train_data['clean_title'])
train_labels = list(train_data['6_way_label'])
## Valid data
valid_news = list(valid_data['clean_title'])
valid_labels = list(valid_data['6_way_label'])
## Test data
test_news = list(test_data['clean_title'])
test_labels = list(test_data['6_way_label'])

## Preprocessing

We define a function to preprocess the data. We remove punctuations and numbers and also multiple spaces.

In [ ]:
def preprocess_text(sen):
    # Remove punctuations and numbers
    sentence = re.sub('[^a-zA-Z]', ' ', sen)

    # Removing multiple spaces
    sentence = re.sub(r'\s+', ' ', sentence)

    return sentence

In [ ]:
# Remove puntuations and numbers and multiple spaces

train_news_clean_1 = []
valid_news_clean_1 = []
test_news_clean_1 = []
# Train
for new in train_news:
    train_news_clean_1.append(preprocess_text(new))
# Validation
for new in valid_news:
    valid_news_clean_1.append(preprocess_text(new))
# Test
for new in test_news:
    test_news_clean_1.append(preprocess_text(new))

Now  we remove stop words and perform lemmatization. We define the function to do that.

In [ ]:
# Initialize  lemmatizer and  stop_words

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english')) 

# Function to remove stopwords and perform lemmatization
def remove_stopwords_lem(text):
    text = word_tokenize(text)
    # Remove stopwords
    text = [word for word in text if word not in stop_words]
    # Lematization
    lemmatized_text = []
    for word in text:
        word1 = lemmatizer.lemmatize(word, pos = "n")
        word2 = lemmatizer.lemmatize(word1, pos = "v")
        word3 = lemmatizer.lemmatize(word2, pos = ("a"))
        lemmatized_text.append(word3)
        
    text_done = ' '.join(lemmatized_text)
    return text_done

And now perform stop-words removal and lemmatization.

In [ ]:
# Stop-words removal and lemmatization
train_stwrd_lem = []
valid_stwrd_lem = []
test_stwrd_lem = []

# Train
for new in train_news_clean_1:
    train_stwrd_lem.append(remove_stopwords_lem(new))
# Validation
for new in valid_news_clean_1:
    valid_stwrd_lem.append(remove_stopwords_lem(new))
# Test
for new in test_news_clean_1:
    test_stwrd_lem.append(remove_stopwords_lem(new))

We train a tokenizer using all the documents and we use the learned vocabulary in order to transform texts into sequences of ID's.

In [ ]:
news_all = train_stwrd_lem + valid_stwrd_lem + test_stwrd_lem

tokenizer = Tokenizer(num_words = 128022)
tokenizer.fit_on_texts(news_all)

# Tokenize news

# Train
train_tokenized = tokenizer.texts_to_sequences(train_stwrd_lem)
# Validation
valid_tokenized = tokenizer.texts_to_sequences(valid_stwrd_lem)
# Test
test_tokenized = tokenizer.texts_to_sequences(test_stwrd_lem)

Obtain the vocabulary length

In [ ]:
print("Vocabulary length: ", len(tokenizer.word_index))

Now we pad the sequences of numbers generated by the tokenizer. But firstly we check how many sequences are shorter than a given length. We start by defining a function that counts the length of each sequence.

In [ ]:
# Function to count the lenght of each sequence
def length_squences(data):
    lengths = []
    for i in range(len(data)):
        lengths.append(len(data[i]))
    return lengths

Now we count the lenghts of all the sequences in the training, validation and test set and we check what % of sequences in the train, validation and test sets are smaller than a given length when they are tokenized.

In [ ]:
length = [10, 15, 20, 25]

# Train set
lengths_train = np.array(length_squences(train_tokenized))
perc_length_train = []
for lgth in length:
   perc_length_train.append( sum(lengths_train < lgth)/len(lengths_train)*100)
print("Pertentages (train):", perc_length_train)

# Validation set 
lengths_valid = np.array(length_squences(valid_tokenized))
perc_length_valid = []
for lgth in length:
   perc_length_valid.append( sum(lengths_valid < lgth)/len(lengths_valid)*100)
print("Pertentages (validation):", perc_length_valid)

# Test set
lengths_test = np.array(length_squences(test_tokenized))
perc_length_test = []
for lgth in length:
   perc_length_test.append( sum(lengths_test < lgth)/len(lengths_test)*100)
print("Pertentages (test):", perc_length_test)

We plot the results.

In [ ]:
import matplotlib.pyplot as plt
# create dataset
bars = [91.42, 91.55, 91.32, 98.45, 98.43, 98.44, 99.49, 99.50, 99.50, 99.81, 99.81, 99.82]
labels = ['< 10', '< 15', '< 20', '< 25']

x_pos_bars = [1, 2, 3, 6, 7, 8, 11, 12, 13, 16, 17, 18]
x_pos_labels = [2, 7, 12, 17]

# Colors and mapping to values
colors = ['b', 'g', 'y']
colors_values = {'Train':'b', 'Validation':'g', 'Test': 'y'}    

labels2 = list(colors_values.keys())
handles = [plt.Rectangle((0,0),1,1, color=colors_values[label]) for label in labels2]
# Make the plot
plt.bar(x_pos_bars, bars,color = colors, label = colors_values)

# Create names on the x-axis
plt.xticks(x_pos_labels, labels)
plt.legend(handles, labels2)
plt.ylim([0, 130])
#plt.xlabel('Metrics')
plt.ylabel('Percentage (%)')
plt.title('% of texts smaller than a given length')
plt.show()

As we can see almost all news are smaller than **15** in length when tokenized so choosing this lenght to pad/truncate the tokenized news will not eliminate any information from the news in almost any case.

In [ ]:
# Pad/truncate the tokenized news

# Train
train_tokenized_pad = pad_sequences(train_tokenized, maxlen = 15, truncating = 'pre', padding = 'pre')
# Validation
valid_tokenized_pad = pad_sequences(valid_tokenized, maxlen = 15, truncating = 'pre', padding = 'pre')
# Test
test_tokenized_pad = pad_sequences(test_tokenized, maxlen = 15, truncating = 'pre', padding = 'pre')

In order to use the data with torch we have to transform the arrays into dataloader objects but first they need to be transformed into tensors.

In [ ]:
# Transform data arrays into tensors

# Train
train_tensor = torch.Tensor(train_tokenized_pad).int()
# Validation
valid_tensor = torch.Tensor(valid_tokenized_pad).int()
# Test
test_tensor =  torch.Tensor(test_tokenized_pad).int()

# Tranform tensors into data loader objects

# Train
train_set = TensorDataset(train_tensor, torch.Tensor(np.array(train_labels)))
trainloader = DataLoader(train_set, batch_size=60)
# Validation
valid_set = TensorDataset(valid_tensor, torch.Tensor(np.array(valid_labels)))
validloader = DataLoader(valid_set, batch_size=60)
# Test
test_set = TensorDataset(test_tensor, torch.Tensor(np.array(test_labels)))
testloader =  DataLoader(test_set, batch_size=60)

### Word embeddings

We will try different word embeddings and select the one which performs better. We create two functions: one to load the word embeddings and the other to create the embedding matrix.

In [ ]:
# Function to load the word embeddings

def load_embedd(filename):
    words = []
    vectors = []
    file = open(filename,'r', encoding="utf8")
    for line in file.readlines():
       row = line.split(' ')
       vocab = row[0]
       embd = row[1:len(row)]
       embd[-1] = embd[-1].rstrip()
       embd = list(map(float,embd)) # convert string to float
       words.append(vocab)
       vectors.append(embd)
    file.close()
    return words,vectors

Function to create the embedding matrix.

In [ ]:
# Function to create the embedding matrix

def embed_matx(word_index, vocab, embeddings, length_vocab, length_embedding):
    embedding_matrix = np.zeros((length_vocab +1, length_embedding))
    for word, i in word_index.items():
        if word in vocab:
            idx = vocab.index(word)
            vector =  embeddings[idx]
            embedding_matrix[i] = vector
        if i == length_vocab:
            break
    return embedding_matrix

#### Glove (300 d)

We use GloVe embeddings of dimension 300.

In [ ]:
vocab_gv_300, vectors_gv_300 = load_embedd(filename = "/content/drive/MyDrive/TFM/glove.6B.300d.txt")

Now we create the embbeding matrix.

In [ ]:
word_index = tokenizer.word_index
# Embedding matrix
embedding_matrix_gv_300 = embed_matx(word_index = word_index, vocab = vocab_gv_300, embeddings = vectors_gv_300, 
                             length_vocab = 117129, length_embedding = 300)

## Models

First we define the CNN that we will use.

In [ ]:
class CNN(nn.Module):
    def __init__(self, nlabels, train_parameters = True, random_embeddings = True): 
        super().__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(num_embeddings =117130, embedding_dim = 300)
        
        if random_embeddings == True:
            self.embedding.weight = nn.Parameter(torch.rand(117130, 300), requires_grad = train_parameters)
        else:
            self.embedding.weight = nn.Parameter(torch.from_numpy(embedding_matrix_gv_300), requires_grad = train_parameters)
            
        # Filters for the CNN    
        self.filter_sizes = [2,3,4,5]
        self.num_filters = 50
        
        # Concolutional layers
        self.convs_concat = nn.ModuleList([nn.Conv2d(1, self.num_filters, (K, 300)) for K in self.filter_sizes])
        
        # Linear layers
        self.linear1 = nn.Linear(200,128)
  
        self.linear2 = nn.Linear(128,nlabels)
    
        self.relu = nn.ReLU()
        
        self.logsoftmax = nn.LogSoftmax(dim=1) 
        
        
    def forward(self, x):
        # Embedding
        x = self.embedding(x)
        # Unsqueeze
        x = x.unsqueeze(1)
        # Convolution
        x = [F.relu(conv(x.float())).squeeze(3) for conv in self.convs_concat]
        # Max-pooling
        x = [F.max_pool1d(i, i.size(2)).squeeze(2) for i in x] 
        x = torch.cat(x, 1)
        # Linear layers
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        x = self.logsoftmax(x) 
        return x

In [ ]:
class CNN_extended(CNN):
    
    def __init__(self,nlabels, train_parameters, random_embeddings, epochs=100, lr=0.001):
        
        super().__init__(nlabels, train_parameters, random_embeddings)  
        
        self.lr = lr #Learning Rate
        
        self.optim = optim.Adam(self.parameters(), self.lr)
        
        self.epochs = epochs
        
        self.criterion = nn.NLLLoss()              
        
        # A list to store the loss evolution along training
        
        self.loss_during_training = [] 

        self.valid_loss_during_training = []
        
    def trainloop(self,trainloader, validloader):
        
        
        # Optimization Loop
        
        for e in range(int(self.epochs)):

            start_time = time.time()
            
            # Random data permutation at each epoch
            
            running_loss = 0.
            
            i = 0
            
            length = 0
            
            accuracies = []
            
            for news, labels in trainloader:             
        
                self.optim.zero_grad()  # Reset gradients
            
                out = self.forward(news.int())

                loss = self.criterion(out,labels.long())
                
                loss.backward()

                running_loss += loss.item()

                self.optim.step()
                
                top_p, top_class = out.topk(1, dim=1)
                
                equals = (top_class == labels.view(news.shape[0], 1))
                
                length += news.shape[0]
                
                accuracies.append(sum(equals)) 
                
                accuracy = sum(accuracies)/length
                
                i += 1
                
                if i%1000 == 0:
                    print(" Train accuracy: ", accuracy)
                
            self.loss_during_training.append(running_loss/len(trainloader))

            # Validation Loss
            
            with torch.no_grad():            
                
                running_loss = 0.
                
                i = 0
                
                length = 0
                
                accuracies = []
                
                for news,labels in validloader:
                    
                    out = self.forward(news.int())

                    loss = self.criterion(out,labels.long())

                    running_loss += loss.item()   
                    
                    top_p, top_class = out.topk(1, dim=1)
                
                    equals = (top_class == labels.view(news.shape[0], 1))
                
                    length += news.shape[0]
                
                    accuracies.append(sum(equals)) 
                
                    accuracy = sum(accuracies)/length
                    
                print(" Validation accuracy: ", accuracy)                    
                      
                self.valid_loss_during_training.append(running_loss/len(validloader))

            if(e % 1 == 0): # Every 10 epochs

                print("Training loss after %d epochs: %f" 
                      %(e,self.loss_during_training[-1]), "Validation loss after %d epochs: %f" %(e,self.valid_loss_during_training[-1]),
                      "Time per epoch: %f seconds"%(time.time() - start_time))

We train the CNN in **4** different scenarios:

  - **Random embeddings** + **Training embeddings**
  - **GloVe embeddings** + **Training embeddings**
  - **Random embeddings** + **Not training embeddings**
  - **GloVe embeddings** + **Not training embeddings**
 
We train the model for several epochs. This is done in order to select the optimal number of epochs, which occurs when the validation loss stops decreasing and starts increasing (Early Stopping). Sometimes the validation loss starts to increase after very few epochs. In those cases we stop the training prematurely since we already know the optimal number of epochs (this is what causes the "KeyboardInterrupt" message in some of the results which can be seen below).

### Random embeddings  + Training embeddings

In [ ]:
# Initialize model
CNN_train_random = CNN_extended(nlabels = 6, epochs=5, lr=0.003, train_parameters = True, random_embeddings = True)
# Train model
CNN_train_random.trainloop(trainloader, validloader)

### GloVe embeddings  + Training embeddings

In [ ]:
# Initialize model
CNN_train_not_random = CNN_extended(nlabels = 6, epochs=4, lr=0.003, train_parameters = True, random_embeddings = False)
# Train model
CNN_train_not_random.trainloop(trainloader, validloader)

### Random embeddings  + Not training embeddings

In [ ]:
# Initialize model
CNN_not_train_random = CNN_extended(nlabels = 6, epochs=9, lr=0.003, train_parameters = False, random_embeddings = True)
# Train model
CNN_not_train_random.trainloop(trainloader, validloader)

### GloVe embeddings  + Not training embeddings

In [ ]:
# Initialize model
CNN_not_train_not_random = CNN_extended(nlabels = 6, epochs=9, lr=0.003, train_parameters = False, random_embeddings = False)
# Train model
CNN_not_train_not_random.trainloop(trainloader, validloader)

## Testing performance

We join the **train** and **validation** datasets and we train the model with this dataset using the optimal number of epochs. Then we evaluate the model with the **test** set. First we need to modify a bit the class used previously.

In [ ]:
class CNN_extended(CNN):
    
    def __init__(self,nlabels, train_parameters, random_embeddings, epochs=100,lr=0.001):
        
        super().__init__(nlabels, train_parameters, random_embeddings)  
        
        self.lr = lr #Learning Rate
        
        self.optim = optim.Adam(self.parameters(), self.lr)
        
        self.epochs = epochs
        
        self.criterion = nn.NLLLoss()               
        
        # A list to store the loss evolution along training
        
        self.loss_during_training = [] 

        self.valid_loss_during_training = []
        
    def trainloop(self,trainloader):
        
        
        # Optimization Loop
        
        for e in range(int(self.epochs)):

            start_time = time.time()
            
            # Random data permutation at each epoch
            
            running_loss = 0.
            
            i = 0
            
            length = 0
            
            accuracies = []
            
            for news, labels in trainloader:             
        
                self.optim.zero_grad()  # Reset gradients
            
                out = self.forward(news.int())

                loss = self.criterion(out,labels.long())
                
                loss.backward()

                running_loss += loss.item()

                self.optim.step()
                
                top_p, top_class = out.topk(1, dim=1)
                
                equals = (top_class == labels.view(news.shape[0], 1))
                
                length += news.shape[0]
                
                accuracies.append(sum(equals)) 
                
                accuracy = sum(accuracies)/length
                
                i += 1
                
                if i%1000 == 0:
                    print(" Train accuracy: ", accuracy)
                
            self.loss_during_training.append(running_loss/len(trainloader))
            end_time = time.time()
            print("Elapsed time: ", end_time-start_time)

                
    def eval_performance(self,dataloader):
        predictions = np.empty((1,1))

        with torch.no_grad():

            for news,labels in dataloader:
                
                logprobs = self.forward(news)  
                top_p, top_class = logprobs.topk(1, dim=1)
                
                top_class_array = np.array(top_class)
                predictions = np.concatenate((predictions, top_class_array), axis = 0)
                
        return predictions[1:]

Now we join the train and validation sets.

In [ ]:
# Join train and validation sequences
train_valid_tokenized_pad = np.concatenate((train_tokenized_pad, valid_tokenized_pad), axis = 0)
# Join train and validation labels
train_valid_labels = np.concatenate((np.array(train_labels), np.array(valid_labels)), axis = 0)

# Create tensor objects

# Train + validation
train_valid_tensor = torch.Tensor(train_valid_tokenized_pad).int()
# Test
test_tensor =  torch.Tensor(test_tokenized_pad).int()

# Tranform tensors into data loader objects

# Train + validation
train_valid_set = TensorDataset(train_valid_tensor, torch.Tensor(np.array(train_valid_labels)))
train_valid_loader = DataLoader(train_valid_set, batch_size=60)
# Test
test_set = TensorDataset(test_tensor, torch.Tensor(np.array(test_labels)))
testloader =  DataLoader(test_set, batch_size=60)

### Random embeddings  + Training embeddings

In [ ]:
# Initialize model
CNN_test_train_random = CNN_extended(nlabels = 6, epochs=1, lr=0.003, train_parameters = True, random_embeddings = True)
# Train model
CNN_test_train_random.trainloop(train_valid_loader)
# Get predictions
predictions1 = CNN_test_train_random.eval_performance(testloader)

In [ ]:
print(classification_report(np.array(test_labels).reshape(len(test_labels),1), predictions1))

In [ ]:
# Classification report without 'true' label
print(classification_report(np.array(test_labels).reshape(len(test_labels),1), predictions1, labels = [1,2,3,4,5]))

In [ ]:
# Confusion matrix
print(confusion_matrix(np.array(test_labels).reshape(len(test_labels),1), predictions1))

### GloVe embeddings  + Training embeddings

In [ ]:
# Initialize model
CNN_test_train_not_random = CNN_extended(nlabels = 6, epochs=1, lr=0.003, train_parameters = True, random_embeddings = False)
# Train model
CNN_test_train_not_random.trainloop(train_valid_loader)
# Get predictions
predictions_2 = CNN_test_train_not_random.eval_performance(testloader)

In [ ]:
print(classification_report(np.array(test_labels).reshape(len(test_labels),1), predictions_2))

In [ ]:
# Classification report without 'true' label
print(classification_report(np.array(test_labels).reshape(len(test_labels),1), predictions_2, labels = [1,2,3,4,5]))

In [ ]:
# Confusion matrix
print(confusion_matrix(np.array(test_labels).reshape(len(test_labels),1), predictions_2))

### Random embeddings  +  Not training embeddings

In [ ]:
# Initialize model
CNN_test_not_train_random = CNN_extended(nlabels = 6, epochs=9, lr=0.003, train_parameters = False, random_embeddings = True)
# Train model
CNN_test_not_train_random.trainloop(train_valid_loader)
# Get predictions
predictions_3 = CNN_test_not_train_random.eval_performance(testloader)

In [ ]:
print(classification_report(np.array(test_labels).reshape(len(test_labels),1), predictions_3))

In [ ]:
# Classification report without 'true' label
print(classification_report(np.array(test_labels).reshape(len(test_labels),1), predictions_3, labels = [1,2,3,4,5]))

In [ ]:
# Confusion matrix
print(confusion_matrix(np.array(test_labels).reshape(len(test_labels),1), predictions_3))

### GloVe embeddings  +  Not training embeddings

In [ ]:
# Initialize model
CNN_test_not_train_not_random = CNN_extended(nlabels = 6, epochs=2, lr=0.003, train_parameters = False, random_embeddings = False)
# Train model
CNN_test_not_train_not_random.trainloop(train_valid_loader)
# Get predictions
predictions_4 = CNN_test_not_train_not_random.eval_performance(testloader)

In [ ]:
print(classification_report(np.array(test_labels).reshape(len(test_labels),1), predictions_4))

In [ ]:
# Classification report without 'true' label
print(classification_report(np.array(test_labels).reshape(len(test_labels),1), predictions_4, labels = [1,2,3,4,5]))

In [ ]:
# Confusion matrix
print(confusion_matrix(np.array(test_labels).reshape(len(test_labels),1), predictions_4))